# Assignment 3: Time Series Classification Part 1


Name: *Pratham Aggarwal* | Github: *prathamaeron* | USC ID: *1537722913*


## Importing the Libraries


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_selection import f_classif


## (a) Loading Data and (b) Train/Test Split

In [12]:
data_dir = Path("../data")
activity_folders = sorted([p for p in data_dir.iterdir() if p.is_dir()])
series_columns = [
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23",
]


def build_feature_frame(files):
    rows = []
    for csv_file in sorted(files):
        activity = csv_file.parent.name
        df = pd.read_csv(
            csv_file,
            comment="#",
            sep=r"[,\s]+",
            engine="python",
            header=None,
            on_bad_lines="skip",
            dtype=str,
        )

        if df.shape[1] < 7:
            continue

        df = df.iloc[:, :7].copy()
        df.columns = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

        row = {"activity": activity, "instance": csv_file.stem}

        for series in series_columns:
            values = pd.to_numeric(df[series], errors="coerce")
            row[f"{series}_min"] = values.min()
            row[f"{series}_max"] = values.max()
            row[f"{series}_mean"] = values.mean()
            row[f"{series}_median"] = values.median()
            row[f"{series}_std"] = values.std(ddof=1)
            row[f"{series}_q1"] = values.quantile(0.25)
            row[f"{series}_q3"] = values.quantile(0.75)

        rows.append(row)

    return pd.DataFrame(rows)


train_files = []
test_files = []

for folder in activity_folders:
    files = sorted(folder.glob("dataset*.csv"))
    for csv_file in files:
        dataset_number = int(csv_file.stem.replace("dataset", ""))
        if folder.name in {"bending1", "bending2"}:
            is_test = dataset_number in {1, 2}
        else:
            is_test = dataset_number in {1, 2, 3}
        if is_test:
            test_files.append(csv_file)
        else:
            train_files.append(csv_file)

all_features = build_feature_frame(train_files + test_files)
train_df = build_feature_frame(train_files)
test_df = build_feature_frame(test_files)

print(f"Training instances: {len(train_df)}")
print(f"Test instances: {len(test_df)}")
print(f"Total instances: {len(all_features)}")
print(all_features.head().to_string(index=False))


Training instances: 69
Test instances: 19
Total instances: 88
activity instance  avg_rss12_min  avg_rss12_max  avg_rss12_mean  avg_rss12_median  avg_rss12_std  avg_rss12_q1  avg_rss12_q3  var_rss12_min  var_rss12_max  var_rss12_mean  var_rss12_median  var_rss12_std  var_rss12_q1  var_rss12_q3  avg_rss13_min  avg_rss13_max  avg_rss13_mean  avg_rss13_median  avg_rss13_std  avg_rss13_q1  avg_rss13_q3  var_rss13_min  var_rss13_max  var_rss13_mean  var_rss13_median  var_rss13_std  var_rss13_q1  var_rss13_q3  avg_rss23_min  avg_rss23_max  avg_rss23_mean  avg_rss23_median  avg_rss23_std  avg_rss23_q1  avg_rss23_q3  var_rss23_min  var_rss23_max  var_rss23_mean  var_rss23_median  var_rss23_std  var_rss23_q1  var_rss23_q3
bending1 dataset1          37.25          45.00       40.624792             40.50       1.476967         39.25         42.00            0.0           1.30        0.358604              0.43       0.322605          0.00          0.50            4.0          29.50       19.040937 

## (c) Feature Extraction

### (c) i. Research

Common time domain features used in time series classification include basic statistics like the minimum, maximum, mean, median, standard deviation, variance, range, skewness, kurtosis, and percentiles (Barandas et al.). People also often use features that measure the size or energy of the signal, such as root mean square (RMS), mean absolute value, total energy, and peak-to-peak distance (Phinyomark et al.). Other popular features describe how the signal changes over time, like the slope, the average difference between neighbouring points, the number of peaks, and the zero-crossing rate (Barandas et al.; Phinyomark et al.). Some more advanced options include autocorrelation and entropy, and tools like tsfresh and catch22 can calculate many of these features automatically (Christ et al.; Lubba et al.).

Works Cited

Barandas, Marília, et al. “TSFEL: Time Series Feature Extraction Library.” SoftwareX, vol. 11, 2020, article 100456, https://doi.org/10.1016/j.softx.2020.100456.

Christ, Maximilian, et al. “Time Series FeatuRe Extraction on Basis of Scalable Hypothesis Tests (tsfresh – A Python Package).” Neurocomputing, vol. 307, 2018, pp. 72–77, https://doi.org/10.1016/j.neucom.2018.03.067.

Lubba, Carl H., et al. “catch22: CAnonical Time-Series CHaracteristics.” Data Mining and Knowledge Discovery, vol. 33, no. 6, 2019, pp. 1821–52, https://doi.org/10.1007/s10618-019-00647-x.

Phinyomark, Angkoon, et al. “Feature Reduction and Selection for EMG Signal Classification.” Expert Systems with Applications, vol. 39, no. 8, 2012, pp. 7420–31, https://doi.org/10.1016/j.eswa.2012.01.102.

### (c) ii. Time-Domain Features

For each instance, I calculate seven statistics for each of the six sensor series: minimum, maximum, mean, median, sample standard deviation, first quartile, and third quartile. This produces 42 features per instance.

In [13]:
feature_columns = [col for col in all_features.columns if col not in {"activity", "instance"}]
feature_summary = all_features[feature_columns].describe().T
feature_summary[["mean", "std", "min", "50%", "max"]].round(4)


,mean,std,min,50%,max
avg_rss12_min,29.0367,9.5700,0.0000,28.6250,48.0000
avg_rss12_max,45.4731,4.3944,30.0000,45.7500,56.2500
avg_rss12_mean,38.9174,5.3357,24.5630,39.4338,48.0042
avg_rss12_median,38.8567,5.4401,24.2500,39.4150,48.0000
avg_rss12_std,2.9890,1.7722,0.0320,3.2127,7.6841
avg_rss12_q1,37.0193,6.1539,23.1875,39.2000,48.0000
avg_rss12_q3,41.1972,5.1389,26.5000,40.2500,54.0000
var_rss12_min,0.0000,0.0000,0.0000,0.0000,0.0000
var_rss12_max,6.2901,5.0627,0.4300,4.5000,17.2400
var_rss12_mean,1.4592,1.5742,0.0072,0.4894,4.5766


### (c) iii. Bootstrap Confidence Intervals

For each feature, I draw 2,000 bootstrap samples with replacement and calculate the sample standard deviation. The 5th and 95th percentiles give a 90% percentile confidence interval.

In [14]:
rng = np.random.default_rng(42)
bootstrap_results = []

for feature in feature_columns:
    values = all_features[feature].dropna().to_numpy()
    base_std = values.std(ddof=1)
    boot_str = np.empty(2000)

    for i in range(2000):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_str[i] = sample.std(ddof=1)

    lower, upper = np.quantile(boot_str, [0.05, 0.95])
    bootstrap_results.append({
        "Feature": feature,
        "Std Dev": base_std,
        "90% CI Lower": lower,
        "90% CI Upper": upper,
    })

bootstrap_table = pd.DataFrame(bootstrap_results).sort_values("Std Dev", ascending=False)
bootstrap_table.round(4)

,Feature,Std Dev,90% CI Lower,90% CI Upper
0,avg_rss12_min,9.5700,8.2270,10.7828
5,avg_rss12_q1,6.1539,5.5854,6.6367
28,avg_rss23_min,6.1240,4.3957,7.4622
33,avg_rss23_q1,6.0965,4.9026,7.1967
31,avg_rss23_median,5.8138,4.5207,6.9229
29,avg_rss23_max,5.7412,4.7594,6.5348
30,avg_rss23_mean,5.6755,4.4077,6.8186
34,avg_rss23_q3,5.5317,4.3897,6.4906
3,avg_rss12_median,5.4401,4.8343,6.0051
2,avg_rss12_mean,5.3357,4.6963,5.9058


I rank training-set features by their one-way ANOVA F-statistic across activity labels and report the top three. Constant features are excluded because they cannot distinguish between activities.

In [15]:
X = train_df[feature_columns]
y = train_df["activity"]

non_constant_features = X.columns[X.nunique(dropna=False) > 1]
X = X[non_constant_features]
f_values, p_values = f_classif(X, y)
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "F-statistic": f_values,
    "p-value": p_values,
}).sort_values("F-statistic", ascending=False)

selected_features = feature_importance.head(3)
selected_features.round(6)

,Feature,F-statistic,p-value
8,var_rss12_mean,901.301852,0.0
12,var_rss12_q3,810.830665,0.0
22,var_rss13_median,795.840520,0.0
